# ID02 - mlops_pipeline_smoke_test

Smoke test end-to-end del pipeline de MLOps del repo: dataset sintético (sin ningún dato sensible ni real) → `dvc add`/`dvc push` → split → **3 clasificadores entrenados y comparados** (cada uno logueado como su propio run de MLflow, mismo experimento) → reporte con una sección por clasificador. Ver `README.md` de este experimento para el detalle completo.

In [ ]:
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

import julieta
from julieta.data.split_data import DataSplitter
from julieta.models.classifiers import BalancedXGBClassifier
from julieta.models.metrics import ClassificationMetrics, julieta_score
from julieta.tracking.mlflow_config import log_run

REPO_ROOT = Path(julieta.__file__).resolve().parents[2]
EXPERIMENT_DIR = Path.cwd().resolve().parent
EXPERIMENT_ID, EXPERIMENT_NAME = EXPERIMENT_DIR.name.split("-", 1)

CONFIG_NAME = "baseline"
with open(EXPERIMENT_DIR / "configs" / f"{CONFIG_NAME}.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)

REPO_ROOT, EXPERIMENT_DIR, EXPERIMENT_ID, EXPERIMENT_NAME

## 1. Generar dataset sintético y subirlo a DVC

In [ ]:
ds_cfg = config["dataset"]
X, y = make_classification(
    n_samples=ds_cfg["n_samples"],
    n_features=ds_cfg["n_features"],
    n_informative=ds_cfg["n_informative"],
    n_redundant=ds_cfg["n_redundant"],
    weights=ds_cfg["class_weights"],
    random_state=ds_cfg["random_state"],
)
feature_names = [f"feature_{i}" for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=feature_names)
df["target"] = y

data_path = REPO_ROOT / "data" / "raw" / "synthetic_smoke_test.csv"
data_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(data_path, index=False)

print(f"dataset: {df.shape}, balance clase positiva: {df['target'].mean():.3f}")

In [ ]:
def run(cmd):
    result = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=True, text=True)
    print(" ".join(cmd), "->", result.returncode)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        result.check_returncode()
    return result


rel_data_path = str(data_path.relative_to(REPO_ROOT))
run([sys.executable, "-m", "dvc", "add", rel_data_path])
run([sys.executable, "-m", "dvc", "push"])

## 2. Split train/test

In [ ]:
X_df = df[feature_names]
y_s = df["target"]

splitter = DataSplitter()
X_train, X_test, y_train, y_test = splitter.split(
    X_df,
    y_s,
    test_size=config["split"]["test_size"],
    stratify=config["split"]["stratify"],
)
X_train.shape, X_test.shape

## 3. Entrenar y evaluar cada clasificador

Mismo train/test para los 3 (comparación justa). Cada uno se loguea como **su propio run de MLflow** (`run_name` = la clave del clasificador en `configs/baseline.yaml`), todos bajo el mismo experimento -- así se ven separados pero agrupados en Azure ML Studio.

In [ ]:
MODEL_CLASSES = {
    "BalancedXGBClassifier": BalancedXGBClassifier,
    "LogisticRegression": LogisticRegression,
    "RandomForestClassifier": RandomForestClassifier,
}


def build_model(model_cfg):
    cls = MODEL_CLASSES[model_cfg["type"]]
    kwargs = {k: v for k, v in model_cfg.items() if k != "type"}
    return cls(**kwargs)


def compute_all_metrics(y_true, y_pred, y_proba):
    cm = ClassificationMetrics(y_true=y_true, y_pred=y_pred, y_proba=y_proba)
    values = cm.get_metrics()
    values["accuracy"] = round(accuracy_score(y_true, y_pred), 2)
    values["julieta_score"] = round(julieta_score(y_true, y_pred), 2)
    return values, cm

In [ ]:
results_dir = EXPERIMENT_DIR / "results"
results_dir.mkdir(parents=True, exist_ok=True)

for model_key, model_cfg in config["models"].items():
    model = build_model(model_cfg)
    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_train_proba = model.predict_proba(X_train)[:, 1]
    y_test_pred = model.predict(X_test)
    y_test_proba = model.predict_proba(X_test)[:, 1]

    train_values, cm_train = compute_all_metrics(y_train, y_train_pred, y_train_proba)
    test_values, cm_test = compute_all_metrics(y_test, y_test_pred, y_test_proba)

    metrics = {f"train_{k}": float(v) for k, v in train_values.items()}
    metrics.update({f"test_{k}": float(v) for k, v in test_values.items()})

    artifact_paths = {}
    for split_name, cm_obj in [("train", cm_train), ("test", cm_test)]:
        fig_cm = cm_obj.plot_confusion_matrix(class_names=["Clase 0", "Clase 1"])
        cm_path = results_dir / f"confusion_matrix_{split_name}_{model_key}.png"
        fig_cm.savefig(cm_path, dpi=150, bbox_inches="tight")
        plt.close(fig_cm)
        artifact_paths[f"confusion_matrix_{split_name}"] = cm_path

        fig_report = cm_obj.plot_classification_report()
        report_path = results_dir / f"classification_report_{split_name}_{model_key}.png"
        fig_report.savefig(report_path, dpi=150, bbox_inches="tight")
        plt.close(fig_report)
        artifact_paths[f"classification_report_{split_name}"] = report_path

    flat_config = {
        **{f"dataset.{k}": v for k, v in config["dataset"].items()},
        **{f"split.{k}": v for k, v in config["split"].items()},
        **{f"model.{k}": v for k, v in model_cfg.items()},
    }

    run_id = log_run(
        experiment_id=EXPERIMENT_ID,
        experiment_name=EXPERIMENT_NAME,
        author="dgrajales",
        config=flat_config,
        metrics=metrics,
        artifacts=[str(p) for p in artifact_paths.values()],
        tags={
            "purpose": "mlops_platform_smoke_test",
            "data_sensitivity": "level_0_synthetic",
            "model_key": model_key,
        },
        run_name=model_key,
    )
    print(f"{model_key} -> run_id={run_id}")